# Dolos — Studio pipeline test (free Colab GPU)

**Run-all notebook.** Turns a text script into a talking-head video on a *lively studio background* with a *natural (non-robotic) voice*, then prints a download link.

It mirrors the Dolos **Studio** worker pipeline (`apps/worker/app/pipeline/studio.py`):

1. **XTTS-v2** — script → `narration.wav` (natural neural voice, clonable) — *replaces Piper*
2. **Talking head** — face photo + audio → `talking_head.mp4`
   - On a **free T4** we use **Wav2Lip** (lightweight, ~4GB, tight lip-sync). On your 24GB A5000, use the optional **Hallo2** cell at the bottom — that's the real Studio model.
3. **Robust Video Matting** — cut the subject out of its frozen background → `fgr.mp4` + `pha.mp4`
4. **ffmpeg composite** — drop the subject onto a looping studio background with a subtle slow-zoom for life → the final video

> **Why Wav2Lip here, not Hallo2:** Hallo2 needs ~24GB VRAM and OOMs on a free T4. The two things this test validates — the **voice** and the **lively-background compositing** — are the exact new `studio.py` code paths and run on any GPU. Wav2Lip is just a light face-driver stand-in so the whole chain completes for free; the "studio life" comes mostly from the background composite. Validate Hallo2 itself on the A5000.

**Before you run:** `Runtime → Change runtime type → Hardware accelerator: GPU`.

- Change what it says: edit **`SCRIPT`** in step 2.
- Clone a voice: set **`VOICE_REF_WAV`** in step 2 to an uploaded ~6s wav.
- Use your own face: step 4 prompts you to upload **`avatar.jpg`**.
- Use a real studio background: set **`STUDIO_BG_URL`** in step 6 to an mp4 loop (e.g. a free Pexels clip).

## 1. Confirm GPU + report VRAM

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  |  VRAM: {vram_gb:.0f} GB')
if vram_gb < 20:
    print('NOTE: <20GB — Wav2Lip (below) is fine; the optional Hallo2 cell will likely OOM here.')

## 2. XTTS-v2 — script → narration.wav  *(the new, natural voice)*

The same engine `studio.py` uses. We run it **first**, before the face model touches the environment, to avoid dependency clashes. Edit `SCRIPT`. To clone a voice, upload a clean ~6s wav and set `VOICE_REF_WAV` to its path.

> If you already ran an *older* `TTS` install in this runtime, do **Runtime → Disconnect and delete runtime** once before running this cell (the old and new packages share the `TTS` module name).

In [ ]:
import os
SCRIPT = "Hey everyone, welcome back to the channel. Today I want to show you something I think you are really going to enjoy, so let us jump right in."
VOICE_REF_WAV = ""          # "" = use a built-in studio speaker; else path to a ~6s reference wav
XTTS_SPEAKER  = "Ana Florence"   # a built-in XTTS-v2 speaker (used when VOICE_REF_WAV is empty)

# XTTS-v2 via `coqui-tts` — the maintained fork of the (abandoned) `TTS`
# package. Same `from TTS.api import TTS` import + same model, but it installs
# and runs on Colab's current Python 3.12 (the old `TTS`/`TTS==0.22.0` does not).
# Remove any old `TTS` first: it shares the `TTS` module namespace and would
# otherwise shadow coqui-tts.
!pip uninstall -y TTS coqui-tts >/dev/null 2>&1
!pip install -q coqui-tts
os.environ["COQUI_TOS_AGREED"] = "1"   # accept the XTTS non-commercial model license non-interactively

# PyTorch >=2.6 defaults torch.load(weights_only=True), which rejects XTTS's
# checkpoint. Allow-list XTTS's config classes so the model loads.
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
from TTS.config.shared_configs import BaseDatasetConfig
torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, XttsArgs, BaseDatasetConfig])

from TTS.api import TTS
AUDIO = "/content/narration.wav"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

kwargs = dict(text=SCRIPT, language="en", file_path=AUDIO)
if VOICE_REF_WAV:
    kwargs["speaker_wav"] = VOICE_REF_WAV      # zero-shot clone
else:
    kwargs["speaker"] = XTTS_SPEAKER           # built-in speaker
tts.tts_to_file(**kwargs)

from IPython.display import Audio
print("narration.wav generated with XTTS-v2")
Audio(AUDIO)

## 3. Install Wav2Lip (light face driver for a free GPU)

Uses the Colab-maintained `justinjohn0306/Wav2Lip` fork. Its checkpoints are hosted as GitHub release assets (stable — no dead Google-Drive links). ~4GB VRAM, quick to set up.

> On the A5000 you'd skip this and use **Hallo2** (optional cell at the bottom) — Wav2Lip is only here so the *voice + compositing* test runs on a free GPU.

In [ ]:
import os

if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip

!apt-get -qq install -y ffmpeg > /dev/null
# Colab-friendly deps (this fork is updated for modern librosa/numpy).
!pip install -q librosa==0.10.2.post1 numba opencv-contrib-python-headless batch-face gdown

# Checkpoints as GitHub release assets on the fork.
!mkdir -p checkpoints face_detection/detection/sfd
!wget -q "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth
!wget -q "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/s3fd.pth" -O face_detection/detection/sfd/s3fd.pth
assert os.path.getsize('checkpoints/wav2lip_gan.pth') > 1_000_000, 'checkpoint download failed'
print('Wav2Lip ready')

## 4. Upload your avatar (`avatar.jpg`)

Run-all pauses here and asks you to pick a file — choose your **`avatar.jpg`** (a clear, front-facing photo; Wav2Lip needs one detectable face).

In [ ]:
import os
AVATAR = '/content/avatar.jpg'

if not os.path.exists(AVATAR):
    from google.colab import files
    print('Select your avatar.jpg (a clear, front-facing photo)…')
    up = files.upload()
    if up:
        with open(AVATAR, 'wb') as f:
            f.write(up[next(iter(up))])

assert os.path.exists(AVATAR), 'No avatar uploaded — re-run this cell and pick avatar.jpg.'
from IPython.display import Image
print('avatar:', AVATAR)
Image(AVATAR, width=256)

## 5. Wav2Lip — photo + audio → talking_head.mp4

In [ ]:
import os
%cd /content/Wav2Lip
TALKING_HEAD = '/content/talking_head.mp4'
!python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face {AVATAR} \
    --audio /content/narration.wav \
    --outfile {TALKING_HEAD} \
    --pads 0 15 0 0 --nosmooth

assert os.path.exists(TALKING_HEAD), 'Wav2Lip produced no output — check the log above.'
print('talking head:', TALKING_HEAD)

## 6. Robust Video Matting — cut the subject out  *(new)*

Exactly what `studio.py._matte()` does: RVM produces a foreground rgb video (`fgr.mp4`) and an alpha/matte video (`pha.mp4`) from the talking head.

In [ ]:
import sys, os, torch

if not os.path.isdir('/content/RobustVideoMatting'):
    !git clone -q https://github.com/PeterL1n/RobustVideoMatting /content/RobustVideoMatting
!pip install -q av
!wget -q -O /content/rvm_mobilenetv3.pth \
    https://github.com/PeterL1n/RobustVideoMatting/releases/download/v1.0.0/rvm_mobilenetv3.pth

sys.path.insert(0, '/content/RobustVideoMatting')
from model import MattingNetwork
from inference import convert_video

model = MattingNetwork('mobilenetv3').eval().cuda()
model.load_state_dict(torch.load('/content/rvm_mobilenetv3.pth', map_location='cuda'))

FGR, PHA = '/content/fgr.mp4', '/content/pha.mp4'
convert_video(
    model,
    input_source=TALKING_HEAD,
    output_type='video',
    output_foreground=FGR,
    output_alpha=PHA,
    seq_chunk=12,
)
print('matte done:', FGR, PHA)

## 7. Composite over a lively studio background  *(new)*

This runs the **exact ffmpeg filtergraph from `studio.py._composite_over_background()`**: matte the subject onto a looping background at 1080p, with the subtle slow zoom (`BG_MOTION`) for life.

By default it **synthesizes** an animated studio-ish background so the notebook is self-contained. For a real look, set `STUDIO_BG_URL` to an mp4 loop (e.g. a free Pexels bokeh/studio clip).

In [ ]:
STUDIO_BG_URL = ""   # "" = synthesize one; else a direct .mp4 URL of a looping studio background
BG = '/content/studio_bg.mp4'

if STUDIO_BG_URL:
    !wget -q -O {BG} "{STUDIO_BG_URL}"
else:
    # Self-contained animated background: a slowly shifting gradient, softened.
    !ffmpeg -y -f lavfi -i \
      "gradients=s=1920x1080:c0=0x0f2027:c1=0x203a43:c2=0x2c5364:c3=0x16222b:x0=300:y0=200:x1=1600:y1=900:nb_colors=4:seed=9:duration=12:speed=0.012" \
      -t 12 -vf "boxblur=10:2" -pix_fmt yuv420p -r 25 {BG}

# Mirrors studio.py: alphamerge the matte, scale subject to 1080, slow-zoom the
# background, overlay centered, keep the talking head's audio.
FILTER = (
    "[1:v][2:v]alphamerge,scale=-1:1080[subj];"
    "[0:v]scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,setsar=1,"
    "zoompan=z='min(zoom+0.0004,1.08)':d=1:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=1920x1080:fps=25[bg];"
    "[bg][subj]overlay=(W-w)/2:(H-h)/2:shortest=1,format=yuv420p[v]"
)
FINAL = '/content/studio_master.mp4'
!ffmpeg -y -stream_loop -1 -i {BG} -i /content/fgr.mp4 -i /content/pha.mp4 -i {TALKING_HEAD} \
    -filter_complex "{FILTER}" -map "[v]" -map "3:a?" \
    -r 25 -c:v libx264 -preset medium -crf 18 -c:a aac -shortest {FINAL}
print('composited:', FINAL)

## 8. Get the video

Uploads the result and prints a URL — **open it in a new browser tab** (Colab's inline player / `files.download()` break when third-party cookies are blocked).

In [ ]:
url = !curl -s -F "reqtype=fileupload" -F "fileToUpload=@{FINAL}" https://catbox.moe/user/api.php
link = url[-1].strip() if url else ''
if link.startswith('http'):
    print('Your video is ready — open this URL in a new browser tab:\n\n   ' + link)
else:
    print('catbox failed (%r), trying tmpfiles.org...\n' % (url,))
    alt = !curl -s -F "file=@{FINAL}" https://tmpfiles.org/api/v1/upload
    print(alt, '\n^ take the URL and insert /dl/ after the domain.')

## Optional — Hallo2 instead of Wav2Lip  *(needs ~24GB VRAM: your A5000, not a free T4)*

This is the real Studio talking-head model (livelier head motion + expression). It will **OOM on a free T4** — run it on the A5000. It replaces steps 3–5: feed it the same `AVATAR` + `narration.wav`, set `TALKING_HEAD` to its output, then **re-run steps 6–8** to matte + composite.

In [ ]:
# Only run on a >=24GB GPU.
if torch.cuda.get_device_properties(0).total_memory / 1e9 < 20:
    raise SystemExit('Skipping Hallo2: needs ~24GB VRAM. Use Wav2Lip above on this GPU.')

if not os.path.isdir('/content/hallo2'):
    !git clone https://github.com/fudan-generative-vision/hallo2 /content/hallo2
%cd /content/hallo2
!pip install -q -r requirements.txt
# Download Hallo2's pretrained models per its README (huggingface-cli / its
# scripts/download) into ./pretrained_models before running.

# Verify these flags against the Hallo2 version you cloned (this is the shape
# studio.py._animate_hallo2 shells out with):
!python scripts/inference_long.py \
    --config ./configs/inference/long.yaml \
    --source_image {AVATAR} \
    --driving_audio /content/narration.wav \
    --output /content/hallo_out

import glob
hv = sorted(glob.glob('/content/hallo_out/**/*.mp4', recursive=True), key=os.path.getmtime)
assert hv, 'No Hallo2 output — check the log and the model download.'
TALKING_HEAD = hv[-1]
print('Hallo2 talking head:', TALKING_HEAD, '\n-> now re-run steps 6, 7, 8 to matte + composite.')

## What this proves & what's next

You validated the **new `studio.py` code paths** that make Dolos look/sound less cheap:

- **XTTS-v2 voice** (step 2) — the fix for the robotic Piper voice, with optional cloning.
- **RVM matte + studio composite** (steps 6–7) — the fix for the frozen background, using the *exact* ffmpeg filtergraph the worker runs, incl. the subtle background motion.

Wav2Lip here is only a light stand-in for the face; the "studio life" comes from the composite. To ship, run the Dolos Celery worker on your **RunPod A5000** with `WORKER_STUB_MODE=false`, `REAL_PIPELINE=studio`, and the `TTS_*` / `HALLO_REPO_PATH` / `RVM_*` / `STUDIO_BG_PATH` vars from `.env.example` — pointed at the same Redis/Postgres/storage as the API. There, use **Hallo2** (the optional cell) as the talking-head model for the fully lively result.